In [ ]:
# 1) JDK + Android SDK
!apt-get update -qq > /dev/null 2>&1 && (apt-get install -y -qq openjdk-21-jdk-headless > /dev/null 2>&1 || apt-get install -y -qq openjdk-17-jdk-headless > /dev/null 2>&1)
!java -version 2>&1 | head -1
import os
if os.path.exists('/usr/lib/jvm/java-21-openjdk-amd64'):
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
else:
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['ANDROID_HOME'] = '/opt/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/opt/android-sdk'
!mkdir -p /opt/android-sdk/cmdline-tools
!curl -sS -o /tmp/ct.zip https://dl.google.com/android/repository/commandlinetools-linux-13114758_latest.zip
!unzip -q -o /tmp/ct.zip -d /opt/android-sdk/cmdline-tools
!mv /opt/android-sdk/cmdline-tools/cmdline-tools /opt/android-sdk/cmdline-tools/latest
!yes | /opt/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null 2>&1
!/opt/android-sdk/cmdline-tools/latest/bin/sdkmanager "platform-tools" "platforms;android-37" > /dev/null 2>&1
print("SDK ready:", os.environ['ANDROID_HOME'])

In [ ]:
# 2) Clone the feature branch
!rm -rf amethyst
!git clone --depth 1 --branch feat/fdroid-libretranslate https://github.com/gemquota/amethyst.git
%cd amethyst
print("cloned")

In [ ]:
# 3) Build BOTH debug APKs + run the F-Droid unit test (single Gradle run)
!./gradlew :amethyst:assemblePlayDebug :amethyst:assembleFdroidDebug :amethyst:testFdroidDebugUnitTest --no-daemon 2>&1 | tail -40

In [ ]:
# 4) Results — copy this cell's output
!echo "=== APKs ==="
!find amethyst/build/outputs/apk -name "*.apk" 2>/dev/null
!echo "=== unit test report ==="
!ls amethyst/build/reports/tests/testFdroidDebugUnitTest/index.html 2>/dev/null && echo "test report exists"
!echo "=== verification ==="
!./gradlew :amethyst:assemblePlayDebug :amethyst:assembleFdroidDebug :amethyst:testFdroidDebugUnitTest --no-daemon 2>&1 | grep -E "BUILD SUCCESSFUL|BUILD FAILED|> Task :amethyst:testFdroidDebugUnitTest|tests completed" | tail -8